In [2]:
import sys
from pathlib import Path
import yaml
import pandas as pd
import xarray as xr
from oceanum.datamesh import Connector

In [3]:
datamesh = Connector()

cycle = pd.Timestamp("2026-01-01 00:00")

ds = datamesh.query(
    datasource="ecmwf_ifs_0p25_oper_sfc",
    variables=["u10", "v10", "t2m", "d2m", "msl", "tp", "ssr", "strd"],
    parameters={"cycle": cycle.strftime("%Y%m%d%H")},          # "2026010100"
    timefilter={"type": "range", "times": [cycle, cycle + pd.Timedelta("12h")]},
    geofilter={"type": "bbox", "geom": [160.5, -52.5, 185.5, -30.5]},
)

print(ds.time.values)                                        # expect 00, 03, 06, 09, 12Z
print(float(ds.longitude.min()), "->", float(ds.longitude.max()))
print(ds["tp"].mean(["latitude", "longitude"]).values)       # accumulation check
print(ds["ssr"].mean(["latitude", "longitude"]).values)
print(ds["strd"].mean(["latitude", "longitude"]).values)

['2026-01-01T00:00:00.000000000' '2026-01-01T03:00:00.000000000'
 '2026-01-01T06:00:00.000000000' '2026-01-01T09:00:00.000000000'
 '2026-01-01T12:00:00.000000000']
160.5 -> 185.5
[          nan 4.3100240e-08 2.9390446e-08 1.9079794e-08 1.9382886e-08]
[          nan 6.5446014e+02 3.9460919e+02 6.0833256e+01 2.4387758e-02]
[      nan 355.77277 353.94644 352.79077 352.92426]


In [4]:
timefilter={"type": "range", "times": [cycle + pd.Timedelta("3h"), cycle + pd.Timedelta("12h")]}

Cycle starts Mar 31 12Z  →  you download Mar 31 15Z, 18Z, 21Z, Apr 1 00Z

In [5]:
# Building the loop 
start = pd.Timestamp("2026-01-01")
ndays = 90

cycles = pd.date_range(start - pd.Timedelta("12h"),          # Dec 31 12Z, so Jan 1 00Z is covered
                       start + pd.Timedelta(f"{ndays}D") - pd.Timedelta("12h"),
                       freq="12h")

print(len(cycles))
print(cycles[0], "...", cycles[-1])

181
2025-12-31 12:00:00 ... 2026-03-31 12:00:00


In [6]:
import os

OUTDIR = "/home/nn411/Projects/OceanAtm/Moana-2.0/atm_test"          # test folder for now
os.makedirs(OUTDIR, exist_ok=True)

VARS = ["u10", "v10", "t2m", "d2m", "msl", "tp", "ssr", "strd"]
BBOX = [160.5, -52.5, 185.5, -30.5]

for cycle in cycles[:3]:                 # only the first 3 for testing
    tag = cycle.strftime("%Y%m%d%H")
    out = f"{OUTDIR}/ifs_{tag}.nc"

    if os.path.exists(out):              # already done, so skip it
        print(tag, "exists, skipping")
        continue

    ds = datamesh.query(
        datasource="ecmwf_ifs_0p25_oper_sfc",
        variables=VARS,
        parameters={"cycle": tag},
        timefilter={"type": "range",
                    "times": [cycle + pd.Timedelta("3h"), cycle + pd.Timedelta("12h")]},
        geofilter={"type": "bbox", "geom": BBOX},
    )

    if ds is None:
        print(tag, "NO DATA")
        continue

    ds.to_netcdf(out)
    print(tag, "saved", ds.sizes["time"], "steps")

2025123112 exists, skipping
2026010100 exists, skipping
2026010112 exists, skipping


In [7]:
ds = xr.open_mfdataset("/home/nn411/Projects/OceanAtm/Moana-2.0/atm_test/ifs_*.nc").sortby("time")
print(ds.time.values)
print("gaps:", (ds.time.diff("time") != pd.Timedelta("3h")).sum().values)

['2025-12-31T15:00:00.000000000' '2025-12-31T18:00:00.000000000'
 '2025-12-31T21:00:00.000000000' '2026-01-01T00:00:00.000000000'
 '2026-01-01T03:00:00.000000000' '2026-01-01T06:00:00.000000000'
 '2026-01-01T09:00:00.000000000' '2026-01-01T12:00:00.000000000'
 '2026-01-01T15:00:00.000000000' '2026-01-01T18:00:00.000000000'
 '2026-01-01T21:00:00.000000000' '2026-01-02T00:00:00.000000000']
gaps: 0


In [8]:
# cell 2: control, after 06Z cycles started
ds2 = datamesh.query(
    datasource="ecmwf_ifs_0p25_oper_sfc",
    variables=["t2m"],
    parameters={"cycle": "2026060106"},       # Jun 1, 06Z
    timefilter={"type": "range", "times": ["2026-06-01T09:00", "2026-06-01T12:00"]},
    geofilter={"type": "bbox", "geom": [160.5, -52.5, 185.5, -30.5]},
)
print(ds2)

<xarray.Dataset> Size: 73kB
Dimensions:    (time: 2, latitude: 89, longitude: 101)
Coordinates:
  * time       (time) datetime64[ns] 16B 2026-06-01T09:00:00 2026-06-01T12:00:00
  * latitude   (latitude) float32 356B -52.5 -52.25 -52.0 ... -31.0 -30.75 -30.5
  * longitude  (longitude) float32 404B 160.5 160.8 161.0 ... 185.0 185.2 185.5
Data variables:
    t2m        (time, latitude, longitude) float32 72kB 281.6 281.5 ... 293.0
Attributes:
    title:                    ECMWF IFS open-data forecast, 0.25 deg, single-...
    institution:              European Centre for Medium-Range Weather Foreca...
    source:                   ECMWF open data (gs://ecmwf-open-data), ifs/0p2...
    license:                  CC-BY-4.0
    forecast_reference_time:  2026-06-01T06:00:00Z
    Conventions:              CF-1.7
    _coordinates:             {"t":"time","x":"longitude","y":"latitude"}
    _datamesh_cache_hash:     de8a79d01fe124bd1a3b0ccd5e5247cb88bce00b32ccc42...
    _request_size:            

In [9]:
# January test: does a 06Z cycle exist before May 2026?
ds3 = datamesh.query(
    datasource="ecmwf_ifs_0p25_oper_sfc",
    variables=["t2m"],
    parameters={"cycle": "2026010106"},       # Jan 1, 06Z
    timefilter={"type": "range", "times": ["2026-01-01T09:00", "2026-01-01T12:00"]},
    geofilter={"type": "bbox", "geom": [160.5, -52.5, 185.5, -30.5]},
)
print(ds3)

DatameshConnectError: Datamesh server error: {"detail":"{\"detail\":\"{\\\"detail\\\":\\\"datasource unavailable: ECMWF open-data forecast cycle 2026-01-01T06:00:00 is not available: not yet complete, outside the archive, or not published as ifs/0p25/oper\\\"}\"}"}
